# "THE PRICE IS RIGHT" Capstone Project

This week - build a model that predicts how much something costs from a description, based on a scrape of Amazon data


A model that can estimate how much something costs, from its description.

# Order of play

DAY 1: Data Curation  
DAY 2: Data Pre-processing  
DAY 3: Evaluation, Baselines, Traditional ML  
DAY 4: Deep Learning and LLMs  
DAY 5: Fine-tuning a Frontier Model  

## DAY 4: Neural Networks and LLMs

Today we'll work from Traditional ML to Neural Networks to Large Language Models!!

In [2]:
# imports

import os
from dotenv import load_dotenv
from huggingface_hub import login
from pricer.evaluator import evaluate
from litellm import completion
from pricer.items import Item
import numpy as np
from tqdm.notebook import tqdm
import csv
from sklearn.feature_extraction.text import HashingVectorizer
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from torch.optim.lr_scheduler import CosineAnnealingLR


In [3]:
LITE_MODE = True

load_dotenv(override=True)
hf_token = os.environ['HF_TOKEN']
login(hf_token, add_to_git_credential=True)

Token has not been saved to git credential helper.
Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Cannot authenticate through git-credential as no helper is defined on your machine.
You might have to re-authenticate when pushing to the Hugging Face Hub.
Run the following command in your terminal in case you want to set the 'store' credential helper as default.

git config --global credential.helper store

Read https://git-scm.com/book/en/v2/Git-Tools-Credential-Storage for more details.


In [4]:
username = "ed-donner"
dataset = f"{username}/items_lite" if LITE_MODE else f"{username}/items_full"

train, val, test = Item.from_hub(dataset)

print(f"Loaded {len(train):,} training items, {len(val):,} validation items, {len(test):,} test items")

README.md:   0%|          | 0.00/735 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/6.07M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/304k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/304k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/20000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Loaded 20,000 training items, 1,000 validation items, 1,000 test items


# Before we look at the Artificial Neural Networks

## There is a different kind of Neural Network we could consider

In [4]:
# Write the test set to a CSV

with open('human_in.csv', 'w', encoding="utf-8") as csvfile:
    writer = csv.writer(csvfile)
    for t in test[:100]:
        writer.writerow([t.summary, 0])

In [5]:
# Read it back in

human_predictions = []
with open('human_out.csv', 'r', encoding="utf-8") as csvfile:
    reader = csv.reader(csvfile)
    for row in reader:
        human_predictions.append(float(row[1]))

In [6]:
def human_pricer(item):
    idx = test.index(item)
    return human_predictions[idx]

In [7]:
human = human_pricer(test[0])
actual = test[0].price
print(f"Human predicted {human} for an item that actually costs {actual}")


Human predicted 120.0 for an item that actually costs 219.0


In [8]:
evaluate(human_pricer, test, size=100)

  0%|          | 0/100 [00:00<?, ?it/s]

$99 $184 $12 $15 $18 $10 $119 $135 $6 $270 $643 $329 $15 $26 $24 $18 $29 $25 $25 $53 $35 $126 $25 $127 $273 $398 $55 $6 $101 $51 $30 $5 $35 $9 $10 $419 $25 $11 $186 $33 $161 $51 $23 $155 $150 $4 $31 $18 $115 $82 $25 $111 $410 $75 $67 $34 $8 $10 $122 $28 $116 $17 $19 $60 $599 $60 $160 $355 $75 $34 $17 $2 $70 $76 $41 $9 $226 $5 $5 $4 $0 $7 $5 $74 $7 $10 $68 $74 $5 $3 $17 $45 $5 $16 $0 $153 $2 $122 $150 $355 

# And now - a vanilla Neural Network

During the remainder of this course we will get deeper into how Neural Networks work, and how to train a neural network.

This is just a sneak preview - let's make our own Neural Network, from scratch, using Pytorch.

Use this to get intuition; it's not important to know all about Neural networks at this point..

In [5]:
# Prepare our documents and prices

y = np.array([float(item.price) for item in train])
documents = [item.summary for item in train]

In [6]:
# Use the HashingVectorizer for a Bag of Words model
# Using binary=True with the CountVectorizer makes "one-hot vectors"

np.random.seed(42)
vectorizer = HashingVectorizer(n_features=5000, stop_words='english', binary=True)
X = vectorizer.fit_transform(documents)

In [7]:
# Define the neural network - here is Pytorch code to create a 8 layer neural network

class NeuralNetwork(nn.Module):
    def __init__(self, input_size):
        super(NeuralNetwork, self).__init__()
        self.layer1 = nn.Linear(input_size, 128)
        self.layer2 = nn.Linear(128, 64)
        self.layer3 = nn.Linear(64, 64)
        self.layer4 = nn.Linear(64, 64)
        self.layer5 = nn.Linear(64, 64)
        self.layer6 = nn.Linear(64, 64)
        self.layer7 = nn.Linear(64, 64)
        self.layer8 = nn.Linear(64, 1)
        self.relu = nn.ReLU()

    def forward(self, x):
        output1 = self.relu(self.layer1(x))
        output2 = self.relu(self.layer2(output1))
        output3 = self.relu(self.layer3(output2))
        output4 = self.relu(self.layer4(output3))
        output5 = self.relu(self.layer5(output4))
        output6 = self.relu(self.layer6(output5))
        output7 = self.relu(self.layer7(output6))
        output8 = self.layer8(output7)
        return output8

In [8]:
# Convert data to PyTorch tensors
X_train_tensor = torch.FloatTensor(X.toarray())
y_train_tensor = torch.FloatTensor(y).unsqueeze(1)

# Split the data into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(X_train_tensor, y_train_tensor, test_size=0.01, random_state=42)

# Create the loader
train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

# Initialize the model
input_size = X_train_tensor.shape[1]
model = NeuralNetwork(input_size)

In [9]:
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Number of trainable parameters: {trainable_params:,}")

Number of trainable parameters: 669,249


In [28]:
# Define loss function and optimizer

loss_function = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# We will do 2 complete runs through the data

EPOCHS = 5

for epoch in range(EPOCHS):
    model.train()
    for batch_X, batch_y in tqdm(train_loader):
        optimizer.zero_grad()

        # The next 4 lines are the 4 stages of training: forward pass, loss calculation, backward pass, optimize
        
        outputs = model(batch_X)
        loss = loss_function(outputs, batch_y)
        loss.backward()
        optimizer.step()

    model.eval()
    with torch.no_grad():
        val_outputs = model(X_val)
        val_loss = loss_function(val_outputs, y_val)

    print(f'Epoch [{epoch+1}/{EPOCHS}], Train Loss: {loss.item():.3f}, Val Loss: {val_loss.item():.3f}')

  0%|          | 0/310 [00:00<?, ?it/s]

Epoch [1/5], Train Loss: 385.918, Val Loss: 16497.967


  0%|          | 0/310 [00:00<?, ?it/s]

Epoch [2/5], Train Loss: 564.509, Val Loss: 16744.752


  0%|          | 0/310 [00:00<?, ?it/s]

Epoch [3/5], Train Loss: 423.575, Val Loss: 16917.877


  0%|          | 0/310 [00:00<?, ?it/s]

Epoch [4/5], Train Loss: 148.496, Val Loss: 16747.193


  0%|          | 0/310 [00:00<?, ?it/s]

Epoch [5/5], Train Loss: 92.285, Val Loss: 16823.342


In [29]:
def neural_network(item):
    model.eval()
    with torch.no_grad():
        vector = vectorizer.transform([item.summary])
        vector = torch.FloatTensor(vector.toarray())
        result = model(vector)[0].item()
    return max(0, result)

In [30]:
evaluate(neural_network, test)

  0%|          | 0/200 [00:00<?, ?it/s]

$145 $72 $15 $19 $75 $198 $28 $21 $15 $175 $446 $183 $6 $356 $153 $5 $138 $10 $119 $59 $54 $25 $123 $181 $240 $351 $352 $32 $2 $21 $22 $94 $67 $6 $87 $44 $21 $99 $51 $87 $178 $1 $46 $123 $143 $33 $74 $69 $7 $56 $2 $60 $85 $45 $104 $67 $20 $49 $67 $22 $149 $16 $13 $158 $482 $89 $101 $290 $119 $176 $18 $69 $0 $20 $24 $14 $162 $7 $1 $5 $65 $64 $53 $60 $52 $78 $37 $23 $102 $111 $30 $83 $6 $3 $28 $43 $10 $97 $161 $304 $16 $62 $19 $82 $28 $0 $103 $292 $14 $18 $47 $58 $61 $15 $54 $249 $292 $177 $56 $12 $31 $408 $70 $11 $27 $79 $14 $57 $12 $57 $30 $58 $5 $57 $158 $17 $24 $88 $10 $69 $41 $134 $21 $269 $144 $69 $68 $320 $143 $40 $1 $62 $29 $77 $6 $45 $105 $42 $69 $24 $107 $8 $4 $75 $224 $18 $111 $32 $10 $12 $24 $4 $68 $2 $41 $44 $34 $51 $46 $43 $479 $15 $227 $99 $59 $62 $5 $30 $6 $9 $27 $28 $8 $7 $8 $7 $25 $35 $12 $1 

# And now - to the frontier!

Let's see how Frontier models do out of the box; no training, just inference based on their world knowledge.

Tomorrow we will do some training.

In [31]:
def messages_for(item):
    message = f"Estimate the price of this product. Respond with the price, no explanation\n\n{item.summary}"
    return [{"role": "user", "content": message}]

In [32]:
print(test[0].summary)

Title: Excess V2 Distortion/Modulation Pedal  
Category: Music Pedals  
Brand: Old Blood Noise  
Description: A versatile pedal offering distortion and three modulation modes—delay, chorus, and harmonized fifths—with full control over signal routing and expression.  
Details: Features include separate gain, tone, and volume controls; time, depth, and volume per modulation; order switching, soft‑touch bypass, and expression jack for dynamic control.


In [33]:
messages_for(test[0])

[{'role': 'user',
  'content': 'Estimate the price of this product. Respond with the price, no explanation\n\nTitle: Excess V2 Distortion/Modulation Pedal  \nCategory: Music Pedals  \nBrand: Old Blood Noise  \nDescription: A versatile pedal offering distortion and three modulation modes—delay, chorus, and harmonized fifths—with full control over signal routing and expression.  \nDetails: Features include separate gain, tone, and volume controls; time, depth, and volume per modulation; order switching, soft‑touch bypass, and expression jack for dynamic control.'}]

In [34]:
# The function for gpt-4.1-nano

def gpt_4__1_nano(item):
    response = completion(model="openai/gpt-4.1-nano", messages=messages_for(item))
    return response.choices[0].message.content

In [35]:
gpt_4__1_nano(test[0])

'$250'

In [36]:
test[0].price

219.0

In [38]:
evaluate(gpt_4__1_nano, test)

  0%|          | 0/200 [00:00<?, ?it/s]

$19 $34 $25 $20 $120 $80 $6 $65 $11 $870 $363 $20 $15 $4 $9 $8 $41 $5 $40 $31 $59 $56 $35 $25 $182 $273 $705 $5 $501 $64 $30 $60 $10 $50 $35 $119 $90 $26 $36 $18 $175 $55 $20 $105 $70 $0 $27 $13 $75 $52 $20 $105 $125 $10 $97 $16 $8 $80 $48 $3 $86 $28 $56 $35 $179 $60 $90 $295 $25 $74 $17 $8 $70 $6 $25 $21 $126 $0 $13 $3 $30 $4 $15 $74 $12 $20 $68 $44 $30 $16 $3 $20 $5 $10 $2 $78 $4 $143 $20 $325 $50 $3 $12 $11 $101 $132 $10 $380 $9 $49 $20 $236 $49 $53 $54 $130 $0 $5 $94 $47 $29 $361 $29 $16 $0 $10 $15 $101 $29 $94 $179 $13 $65 $5 $85 $10 $55 $0 $78 $62 $16 $150 $30 $9 $124 $118 $25 $340 $15 $13 $3 $144 $12 $4360 $3 $129 $31 $36 $70 $5 $211 $17 $8 $2 $340 $2 $752 $25 $5 $5 $5 $3 $120 $8 $52 $201 $3 $57 $34 $13 $546 $25 $150 $99 $0 $3 $73 $17 $10 $2 $0 $69 $25 $11 $50 $40 $10 $120 $21 $1 

In [41]:
def claude_opus_4_6(item):
    response = completion(model="anthropic/claude-opus-4-6", messages=messages_for(item))
    return response.choices[0].message.content

In [42]:
evaluate(claude_opus_4_6, test, size=10)

  0%|          | 0/10 [00:00<?, ?it/s]

$20 $4 $15 $35 $20 $30 $94 $70 $10 $45 

In [43]:
def gemini_3_1_pro_preview(item):
    response = completion(model="gemini/gemini-3-1-pro-preview", messages=messages_for(item), reasoning_effort='low')
    return response.choices[0].message.content

In [ ]:
evaluate(gemini_3_1_pro_preview, test, size=10, workers=2)

In [39]:
def gemini_2__5_flash_lite(item):
    response = completion(model="gemini/gemini-2.5-flash-lite", messages=messages_for(item))
    return response.choices[0].message.content

In [40]:
evaluate(gemini_2__5_flash_lite, test)

  0%|          | 0/200 [00:00<?, ?it/s]

$30 $134 $15 $50 $30 $130 $69 $65 $1 $70 $313 $20 $25 $4 $19 $12 $71 $5 $240 $31 $14 $24 $20 $75 $102 $253 $145 $5 $251 $65 $15 $15 $90 $50 $5 $31 $20 $31 $34 $13 $135 $35 $0 $20 $100 $5 $10 $3 $75 $82 $28 $95 $125 $0 $67 $14 $8 $50 $32 $13 $116 $33 $16 $60 $129 $30 $60 $295 $5 $104 $17 $8 $70 $6 $15 $11 $126 $0 $8 $7 $30 $0 $0 $74 $8 $10 $168 $44 $35 $1 $3 $55 $15 $5 $0 $103 $11 $37 $120 $325 $20 $33 $3 $41 $51 $32 $17 $350 $14 $49 $10 $161 $29 $43 $54 $105 $7 $0 $64 $247 $9 $211 $50 $16 $0 $5 $10 $50 $1 $74 $129 $13 $15 $5 $135 $5 $55 $0 $3 $22 $1 $150 $15 $0 $56 $18 $5 $240 $185 $3 $1 $104 $12 $20 $6 $71 $26 $41 $0 $10 $89 $13 $28 $2 $90 $7 $752 $15 $25 $5 $9 $3 $221 $10 $27 $71 $3 $18 $76 $33 $4 $10 $250 $26 $40 $3 $58 $17 $15 $12 $0 $24 $20 $181 $25 $49 $10 $30 $4 $1 

In [ ]:

def grok_4__1_fast(item):
    response = completion(model="xai/grok-4-1-fast-non-reasoning", messages=messages_for(item), seed=42)
    return response.choices[0].message.content

In [ ]:
evaluate(grok_4__1_fast, test)

In [47]:
# The function for gpt-5.1

def gpt_5__4(item):
    response = completion(model="gpt-5.4-2026-03-05", messages=messages_for(item), reasoning_effort='none', seed=42)
    return response.choices[0].message.content


In [48]:
evaluate(gpt_5__4, test, size=10)

  0%|          | 0/10 [00:00<?, ?it/s]

$10 $104 $20 $10 $10 $170 $54 $70 $11 $141 